In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN,Embedding,Dense,Dropout


In [ ]:
VOCAB_SIZE=10000
MAX_LENGTH=200
EMBEDDING_DIM = 32
RNN_UNITS = 32

In [ ]:
(x_train,y_train),(x_test,y_test) = imdb.load_data(num_words=VOCAB_SIZE)


17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
x_train = pad_sequences(
    x_train,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

x_test = pad_sequences(
    x_test,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

In [ ]:
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM,input_length=MAX_LENGTH),
    SimpleRNN(RNN_UNITS),
    Dropout(0.2),
    Dense(1,activation="sigmoid")
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(x_train,y_train,epochs=5,batch_size=128,validation_split=0.2)

Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 15s 83ms/step - accuracy: 0.4980 - loss: 0.6961 - val_accuracy: 0.5156 - val_loss: 0.6933
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 12s 75ms/step - accuracy: 0.5990 - loss: 0.6618 - val_accuracy: 0.5030 - val_loss: 0.6972
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 21s 77ms/step - accuracy: 0.6815 - loss: 0.5730 - val_accuracy: 0.5118 - val_loss: 0.7559
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 12s 74ms/step - accuracy: 0.7440 - loss: 0.4575 - val_accuracy: 0.5154 - val_loss: 0.8175
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 12s 74ms/step - accuracy: 0.7978 - loss: 0.3760 - val_accuracy: 0.5118 - val_loss: 0.9160


In [ ]:
word_index = imdb.get_word_index()


1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
word_index = {
    word: index+3 for word,index in word_index.items()
}

In [ ]:
word_index["<PAD>"]=0
word_index["START>"]=1
word_index["UNK"]=2
word_index["UNUSED"]=3

def encode_review(review):
  words = review.lower().split()
  encode_words = []
  for word in words:
    word = word.strip(":.,?\"{}[]()") # Fixed assignment for strip
    if word in word_index:
      encode_words.append(word_index[word])
    else:
      encode_words.append(2)  # UNK token

  encoded_words = pad_sequences(
      [encode_words],
      maxlen=MAX_LENGTH,
      padding="post",
      truncating="post"
  )
  return encoded_words


In [ ]:
def predict_review(review):
  encoded_review = encode_review(review)
  probability = model.predict(encoded_review,verbose=0)[0][0]
  if probability >0.5:
    sentiment="Positive"
  else:
    sentiment = "Negative"
  return sentiment

In [ ]:
input_review = input("Enter the review:")
print(predict_review(input_review))

Enter the review:this movie is trash and boring
Positive
